# 获取CIC-IDS2017采样个数对应识别bad cluster的概率

In [10]:
'''
Description: 一键评估
'''
import torch
from torch_geometric.loader import DataLoader
import pandas as pd
from incdbscan import IncrementalDBSCAN
import time

import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
import ast
import wandb
import sys
import os
# 以当前 notebook 的工作目录为基准
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)
from model.gps_gobal_model import NNConv_model
from get_base_pattern import get_base_pattern,base_pattern_parse
import get_data
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# 获取CIC-IDS2017随采样数变化识别不良集群的概率
# 全局超参数设定
coo_graph_train_path = "../..//dataset/graph-format-data/graph_train.csv"
coo_graph_test_path = "../..//dataset/graph-format-data/graph_test.csv"
# 基础模式应包含的最小安全事件数量
min_base_pattern_graph_num = 1
node_out_feature = 30
graph_out_feature = 2


# 标签处理函数
def parse_and_strip_benign(label_str):
    label_set = ast.literal_eval(label_str)  # 安全地解析字符串为集合
    new_set = label_set - {'BENIGN'}

    return str(new_set) if new_set else str(label_set)

## 一、数据的联合，embedding data, coo graph data, base pattern datadef get_base_pattern_data(coo_graph_df, embedding_data_df, encoded_signature_df):
def get_base_pattern_data(coo_graph_df, embedding_data_df):
    """
    该函数用于获取图的基本模式画像，并将其与嵌入数据和编码签名数据合并。

    参数:
        coo_graph_df (DataFrame): 包含图的COO格式数据的DataFrame。
        embedding_data_df (DataFrame): 包含嵌入数据的DataFrame。
    返回:
        DataFrame: 合并后的DataFrame，包含图的基本模式画像、嵌入数据和编码签名数据。
    """
    encoded_signature_path = "../..//dataset/match_knowledge/encoded_signature.csv"
    encoded_signature_df = pd.read_csv(encoded_signature_path)
    # 获取基本模式画像等图基本信息
    coo_graph_df['base_profile'] = ''
    coo_graph_df['base_pattern_parse'] = ''
    coo_graph_df['base_topologic'] = ''

    for i in range(len(coo_graph_df)):
        now_graph = coo_graph_df.iloc[i]
        # 获取基本模式画像
        base_pattern = get_base_pattern(now_graph)
        # 解析基本模式画像
        parse = base_pattern_parse(base_pattern, encoded_signature_df)
        # 中心节点个数，拓扑类型
        base_topologic = (base_pattern[0], base_pattern[3])
        # 将基本模式画像、解析结果和拓扑信息添加到DataFrame中
        coo_graph_df.loc[i, 'base_profile'] = str(base_pattern)
        coo_graph_df.loc[i, 'base_pattern_parse'] = str(parse)
        coo_graph_df.loc[i, 'base_topologic'] = str(base_topologic)

    # 合并COO图数据和嵌入数据
    graph_all_data = pd.merge(coo_graph_df, embedding_data_df, on=['id'], how='left')
    # 统计每个簇的告警数量
    edges_label = graph_all_data['edge_label'].to_list()
    warn_num_list = []
    for one_edge in edges_label:
        one_edge_label = eval(one_edge)
        warn_num = sum(one_edge_label.values())
        warn_num_list.append(warn_num)
    graph_all_data['warn_num'] = warn_num_list

    # 对label进行转换
    graph_all_data['group_label'] = graph_all_data['group_label'].apply(parse_and_strip_benign)
    condition = (graph_all_data['group_label'] == "{'BENIGN'}")

    # group_label_easy仅记录二分类结果
    graph_all_data['group_label_easy'] = "{'BENIGN'}"
    graph_all_data.loc[~condition,'group_label_easy'] = "{'High Risk'}"
    # risk_level记录二分类结果
    graph_all_data['risk_level'] = 0
    graph_all_data.loc[~condition,'risk_level'] = 1
    label_map = {
    "{'BENIGN'}": 0,
    "{'Bot'}": 1,
    "{'PortScan'}": 2,
    "{'DDoS'}": 3,
    "{'SSH-Patator'}": 4,
    "{'DoS Hulk'}": 5,
    "{'Web Attack ?Brute Force'}": 6,
    "{'Web Attack ?XSS'}": 7,
    "{'DoS slowloris'}": 8,
    "{'Infiltration'}": 9,
    "{'DoS Slowhttptest'}": 10,
    "{'DoS GoldenEye'}": 11,
    "{'FTP-Patator'}": 12,
    "{'Web Attack ?Sql Injection'}": 13
    }
    # label_id记录多分类结果
    graph_all_data['label_id'] = graph_all_data['group_label'].map(label_map)
    return graph_all_data


def one_base_cluster(one_base_graph_data,IncrementalDBSCAN,base_graphId:dict,base_embedding:dict,base_risk:dict):
    """对一个base画像内的告警簇进行聚类
        
    Args:
        one_base_graph_data: 一个base画像内所有的告警簇数据，格式如graph_data_all一般
        IncrementalDBSCAN: 一个base画像对应的增量聚类器
    """
    one_base_graph_data = one_base_graph_data.copy()
    base_pattern_parse = one_base_graph_data['base_pattern_parse'].unique()[0]
    embedding_reduce = one_base_graph_data['embedding_reduce']
    embedding_reduce_list = embedding_reduce.to_list()
    embedding_reduce_array = embedding_reduce_list
    
    IncrementalDBSCAN.insert(embedding_reduce_array)
    cluster_labels = IncrementalDBSCAN.get_cluster_labels(embedding_reduce_array)
    cluster_labels = list(cluster_labels)
    
    base_graphId[base_pattern_parse] = one_base_graph_data['id'].to_list()
    base_risk[base_pattern_parse] = one_base_graph_data['risk_level'].to_list()
    base_embedding[base_pattern_parse] = embedding_reduce_array

    ############################2返回clsuter与风险等级分布############################
    #                                                                                #
    #                         clsuter与风险等级分布                                  #
    #                                                                                #
    ##################################################################################
    # 将警告级别设置为 'raise' 以外的选项
    #pd.set_option('mode.chained_assignment', None)
    # cluster_labels是一个列表，要赋值给DataFrame格式one_base_graph_data的一列
    assert len(cluster_labels) == len(one_base_graph_data), "cluster_labels 长度与 DataFrame 行数不一致"
    one_base_graph_data.loc[:, 'HDBSCAN_LABEL'] = cluster_labels
    # 使用 groupby 和 agg 进行分组、计数和求和
    cluster_distri = one_base_graph_data.groupby(['HDBSCAN_LABEL', 'group_label_easy']) \
        .agg({'id': 'count', 'warn_num': 'sum'}) \
        .reset_index() \
        .rename(columns={'id': 'graph_count'})
    # 对id列进行分组，并组成列表
    grouped_id = one_base_graph_data.groupby(['HDBSCAN_LABEL', 'group_label_easy'])['id'].apply(list).reset_index()

    # 将两个DataFrame进行合并
    result_df = pd.merge(cluster_distri, grouped_id, on=['HDBSCAN_LABEL', 'group_label_easy'])
    result_df['base_pattern_parse'] = base_pattern_parse
    result_df['HDBSCAN_LABEL'] = cluster_distri['HDBSCAN_LABEL'].astype(int)
    
    return result_df

from math import comb
# 获取采样个数
def get_supsicous_prob(data:pd.DataFrame, k):
    """ 获取单个集群识别为supsicous的概率
    
    Args:
        data: 单个集群对应的数据表
        k: 采样个数
        
    Return:
        find_supsicous_prob: 识别该集群为supsicous的概率的概率
    """
    find_supsicous_prob = 0
    no_find_supsicous_prob = 0
    # 如果风险等级数量最多的个数小于k，则一定能识别为supsicous
    #print(data)
    group_label_max_num = data['graph_count'].max()
    graph_sum_num = data['graph_count'].sum()
    if group_label_max_num < k:
        find_supsicous_prob = 1
        return find_supsicous_prob
    # 否则在所有的风险等级分类中，找出个数>=k的，然后计算
    else:
        for index_count in range(len(data)):
            graph_count = data.loc[index_count,'graph_count']
            if graph_count >= k:
                no_find_supsicous_prob += comb(graph_count, k)/comb(graph_sum_num, k)
        find_supsicous_prob = 1 - no_find_supsicous_prob
        return find_supsicous_prob

def get_sample_num(unique_values_count:pd.DataFrame,clsuter_ok_distri,cluster_single_risk_percent):
    """ 获取采样数量对应识别概率

    Args:
        data: 单个集群对应的数据表
        k: 采样个数

    Return:
        sample_num: 采样个数
    """
    overall_supsicous_prob_list = []
    for n_samples in range(0,32):
        supicious_clusters = unique_values_count[unique_values_count['group_label_num']>1]
        supicious_clusters = supicious_clusters.reset_index(drop=True)
        supicious_cluster_name = supicious_clusters['cluster_name'].unique()

        supsicous_prob_list = []
        for cluster_name in supicious_cluster_name:
            cluster_data = clsuter_ok_distri.loc[clsuter_ok_distri['cluster_name']==cluster_name]
            cluster_data = cluster_data.reset_index(drop=True)
            supsicous_prob = get_supsicous_prob(cluster_data,k=n_samples)
            supsicous_prob_list.append(supsicous_prob)
    
        supsicous_prob_mean = sum(supsicous_prob_list)/len(supsicous_prob_list)
        overall_supsicous_prob = supsicous_prob_mean * (1-cluster_single_risk_percent) + 1*cluster_single_risk_percent
        overall_supsicous_prob_list.append(overall_supsicous_prob)
    return overall_supsicous_prob_list


In [ ]:
def evaluation(train_embedding_path, eps=0.10, min_pts=3):
    """评估函数
    Args:
        train_embedding_path (str): 训练embedding数据的所在路径, 为csv格式
        test_embedding_path (str): 测试embedding数据的所在路径, 为csv格式
        model_param (str): 模型参数, 默认为''
        eps (float): 增量聚类的eps参数
        min_pts (int): 增量聚类的min_pts参数
    Returns:
        None, 但记录此次测试结果信息
    """
    coo_graph_df = pd.read_csv(coo_graph_train_path)
    embedding_data_df = pd.read_csv(train_embedding_path)
    embedding_data_df['embedding_data'] = embedding_data_df['embedding_data'].apply(ast.literal_eval)
    graph_all_data = get_base_pattern_data(coo_graph_df,embedding_data_df)
    graph_all_data['embedding_reduce'] = graph_all_data['embedding_data']

    ### 满足basic pattern基本数量,筛选graph data, 生成graph_all_data_select
    pattern_classification = graph_all_data.groupby(['base_pattern_parse'])['id'].apply(list).reset_index()
    pattern_classification = pattern_classification.rename(columns={'id': 'id_list'})
    pattern_classification['count'] = pattern_classification['id_list'].apply(len)
    limit_base_pattern = pattern_classification.loc[pattern_classification['count']>=min_base_pattern_graph_num]
    graph_all_data_select = pd.merge(graph_all_data, limit_base_pattern, on=['base_pattern_parse'])
    base_profile_support  = graph_all_data_select['base_pattern_parse'].unique()

    cluster_distri_list = []

    ### 增量聚类
    base_dbscan = {}
    base_graphId = {}
    base_embedding = {}
    base_risk = {}
    for i,one_base in enumerate(base_profile_support):
        one_base_graph_data = graph_all_data_select[graph_all_data_select['base_pattern_parse'] == one_base]
        base_dbscan[one_base] = IncrementalDBSCAN(eps=eps, min_pts=min_pts)
        cluster_distri= one_base_cluster(one_base_graph_data,base_dbscan[one_base],base_graphId,base_embedding,base_risk)
        cluster_distri_list.append(cluster_distri)

    # 使用pd.concat将所有DataFrame组合在一起
    cluster_distri_res = pd.concat(cluster_distri_list, ignore_index=True)
    
    ### 手动阶段评估
    # 统计噪声点（-1）的占比
    clsuter_ok_distri = cluster_distri_res[cluster_distri_res['HDBSCAN_LABEL']!=-1]
    clsuter_ok_distri = clsuter_ok_distri.reset_index(drop=True)
    cluster_noise_distri = cluster_distri_res[cluster_distri_res['HDBSCAN_LABEL']==-1]
    cluster_noise_num, subgraph_noise_num, alert_noise_num = len(cluster_noise_distri), cluster_noise_distri['graph_count'].sum(), cluster_noise_distri['warn_num'].sum()

    # 计算覆盖率
    no_select_subgraph = len(graph_all_data) -  len(graph_all_data_select)
    subgraph_coverage = 1 - (no_select_subgraph + subgraph_noise_num) / len(graph_all_data)
    # 计算单一风险等级的占比
    clsuter_ok_distri['cluster_name'] = list(zip(clsuter_ok_distri['base_pattern_parse'],clsuter_ok_distri['HDBSCAN_LABEL']))
    unique_values_count = clsuter_ok_distri.groupby(['cluster_name']) \
        .agg({'group_label_easy': ['nunique', list], 'graph_count':'sum', 'warn_num':'sum'}) \
        .reset_index()
    unique_values_count.columns = ['cluster_name', 'group_label_num', 'group_label_list', 'graph_count', 'warn_num']
    one_single_risk_data = unique_values_count[unique_values_count['group_label_num']==1]
    cluster_single_risk_num = len(one_single_risk_data)
    cluster_single_risk_percent = cluster_single_risk_num / len(unique_values_count)
    subgraph_single_risk_num = one_single_risk_data['graph_count'].sum()
    subgraph_single_risk_percent = subgraph_single_risk_num / unique_values_count['graph_count'].sum()
    alert_single_risk_num = one_single_risk_data['warn_num'].sum()
    alert_single_risk_percent = alert_single_risk_num / unique_values_count['warn_num'].sum()
    print('---------------------------------Workload reduction RESULTS----------------------------------------------------')
    print("唯一风险等级集群占比: {:.2%}, 对应簇占比: {:.2%}, 告警占比{:.2%}".format(cluster_single_risk_percent,subgraph_single_risk_percent,alert_single_risk_percent))

    sample_prob_list = get_sample_num(unique_values_count,clsuter_ok_distri,cluster_single_risk_percent)
    print(f"采样数量对应概率为: {sample_prob_list}")
    return sample_prob_list

In [12]:
# model_select_ver = 'only_pre'
# model_select_ver = 'random'
# model_select_ver = 'best'
model_select_ver = 'pre_and_tune'
min_pts = 2
eps = 0.02
ratio_list = [0.10]
#ratio_list = [0]
for ratio in ratio_list:
    fine_tune_num = int(ratio * 100)
    # 模型选择
    if model_select_ver == 'only_tune':
        model_path = f"../..//checkpoints/fine-tune-model2/{fine_tune_num}/gps_model_only_tune.pt"
    elif model_select_ver == 'only_pre':
        model_path = f"../..//checkpoints/pre-train-model/gps_gobal_model.pt"

    elif model_select_ver == 'pre_and_tune':
        model_path = f"../..//checkpoints/fine-tune-model2/{fine_tune_num}/gps_model.pt"
    
    elif model_select_ver == 'random':
        model_path = ''

    
    elif model_select_ver == 'best':
        model_path = ''

    train_embedding_pre_path = f"../..//src/evluation/graph_embeding_data/{model_select_ver}/train/{fine_tune_num}"
    if not os.path.exists(train_embedding_pre_path):
        os.makedirs(train_embedding_pre_path)
    train_embedding_path = f"{train_embedding_pre_path}/train_embedding.csv"

    test_embedding_pre_path = f"../..//src/evluation/graph_embeding_data/{model_select_ver}/test/{fine_tune_num}"
    if not os.path.exists(test_embedding_pre_path):
        os.makedirs(test_embedding_pre_path)
    test_embedding_path = f"{test_embedding_pre_path}/test_embedding.csv"

    # 获取graph_embedding
    get_embedding_start_time = time.time()
    if not os.path.exists(train_embedding_path):
        print("get_train_embedding")
        data_train = get_data.get_all_data(coo_graph_train_path,PE=True)
        get_embedding(data_train, model_path, train_embedding_path)
    if not os.path.exists(test_embedding_path):
        data_test = get_data.get_all_data(coo_graph_test_path,PE=True)
        get_embedding(data_test, model_path, test_embedding_path)
    get_embedding_end_time = time.time()
    print(f"get_embedding time: {get_embedding_end_time - get_embedding_start_time}s")
    sample_prob_list = evaluation(train_embedding_path, eps=eps, min_pts=min_pts)


get_embedding time: 7.62939453125e-06s
---------------------------------Workload reduction RESULTS----------------------------------------------------
唯一风险等级集群占比: 99.58%, 对应簇占比: 99.51%, 告警占比99.97%
采样数量对应概率为: [0.9916550764951322, 0.9958275382475661, 0.9982665134748342, 0.998790590796396, 0.9988510612565762, 0.9989115317167564, 0.9989720021769366, 0.9990324726371168, 0.999092943097297, 0.9991534135574772, 0.9992138840176574, 0.9992743544778376, 0.9993348249380177, 0.9993952953981979, 0.9994557658583781, 0.9995162363185583, 0.9995767067787386, 0.9996371772389188, 0.999697647699099, 0.9997581181592792, 0.9998185886194594, 0.9998790590796396, 0.9999395295398198, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [9]:
sample_prob_list

[0.9958275382475661,
 0.9982665134748342,
 0.998790590796396,
 0.9988510612565762,
 0.9989115317167564,
 0.9989720021769366,
 0.9990324726371168,
 0.999092943097297,
 0.9991534135574772,
 0.9992138840176574,
 0.9992743544778376,
 0.9993348249380177,
 0.9993952953981979,
 0.9994557658583781,
 0.9995162363185583,
 0.9995767067787386,
 0.9996371772389188,
 0.999697647699099,
 0.9997581181592792,
 0.9998185886194594,
 0.9998790590796396,
 0.9999395295398198,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0]